# Data Cleaning of U.S. Businesses Dataset

## 1. Introduction

This project demonstrates a structured data-cleaning workflow on a sample of U.S. business records.

The cleaning logic is implemented in reusable functions inside the `src/data_cleaner` package, while this notebook documents the cleaning process and validates the results.

## 2. Imports

In [70]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
PROCESSED_DATA_DIR = DATA_DIR / "processed"
EXTERNAL_DATA_DIR = DATA_DIR / "external"

SRC_DIR = PROJECT_ROOT / "src"

sys.path.append(str(SRC_DIR))

In [71]:
import pandas as pd

from data_cleaner.functions import (
    clean_business_name,
    clean_business_type,
    clean_state_postal_abbr,
    clean_address,
    clean_us_city,
    clean_zip_code,
    validate_zip_state
)

## 3. Loading the Dataset

In [72]:
df = pd.read_csv(RAW_DATA_DIR / "us_businesses_data.csv")

## 4. Initial Data Inspection

### 4.1 Dataset Overview

In [73]:
df.shape

(1000, 18)

In [74]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   name               1000 non-null   str    
 1   business_type      1000 non-null   str    
 2   state_registered   1000 non-null   str    
 3   street_registered  630 non-null    str    
 4   city_registered    632 non-null    str    
 5   zip5_registered    613 non-null    float64
 6   state_physical     546 non-null    str    
 7   street_physical    598 non-null    str    
 8   city_physical      770 non-null    str    
 9   zip5_physical      485 non-null    float64
 10  filing_number      989 non-null    str    
 11  public             811 non-null    str    
 12  naics_2017         24 non-null     float64
 13  ein                0 non-null      float64
 14  sic4               0 non-null      float64
 15  parent             0 non-null      float64
 16  website            4 non-null      s

In [75]:
df.head()

,name,business_type,state_registered,street_registered,city_registered,zip5_registered,state_physical,street_physical,city_physical,zip5_physical,filing_number,public,naics_2017,ein,sic4,parent,website,duns
0,! ! ! APPLE IPAD & ANDROID TABLET TUTORING ! ! !,CORPORATION,CA,"10100 SANTA MONICA BLVD, # 300",LOS ANGELES,90067.0,CA,"10100 SANTA MONICA BLVD, # 300",LOS ANGELES,90067.0,C3110952,N,NaN,NaN,NaN,NaN,NaN,NaN
1,! ! 0 1 AGLC DUI AND DEFENSIVE DRIVING LLC,LLC,GA,"3235 SATELLITE BLVD., BLD. 400 STE.300",DULUTH,30032.0,GA,"3235 SATELLITE BLVD., BLD. 400",DULUTH,30096.0,15094100,N,541690.0,NaN,NaN,NaN,NaN,NaN
2,! ! 0 1 ATLANTA AREA DUI AND DRIVING SCHOOL LLC,LLC,GA,4493 ASHEILA COURT,LILBURN,30047.0,GA,"3845 NORTH DRUID HILLS RD, SUITE 106",DECATUR,30033.0,10078480,N,NaN,NaN,NaN,NaN,NaN,NaN
3,! 0 $294 1 DAY MT LLC,LLC,MT,NaN,NaN,59901.0,NaN,NaN,NaN,NaN,C267413,N,NaN,NaN,NaN,NaN,NaN,NaN
4,"! ACHIEVE SUCCESS, LLC",LLC,FL,100 NE 95 ST,MIAMI SHORES,33138.0,FL,100 NE 95 ST,MIAMI SHORES,33138.0,L09000025190,N,NaN,NaN,NaN,NaN,http://www.achievesuccess1.com/,NaN


### 4.2 Duplicates

In [76]:
df.duplicated().sum()

np.int64(0)

### 4.3 Missing Values

The dataset contains substantial missingness in several columns.

Columns with extremely high proportions (95–100%) of missing values (`ein`, `sic4`, `duns`, `parent`, `website`, `naics_2017`) were considered for removal when they provided insufficient information for the cleaning task.

In [77]:
null_percentage = df.isna().mean() * 100

print(null_percentage.sort_values(ascending=False))

parent               100.0
sic4                 100.0
duns                 100.0
ein                  100.0
website               99.6
naics_2017            97.6
zip5_physical         51.5
state_physical        45.4
street_physical       40.2
zip5_registered       38.7
street_registered     37.0
city_registered       36.8
city_physical         23.0
public                18.9
filing_number          1.1
name                   0.0
business_type          0.0
state_registered       0.0
dtype: float64


In [78]:
df.drop(columns=["parent", "sic4", "duns", "ein", "website", "naics_2017"], inplace=True)

## 5. Data Cleaning

### 5.1 General Formatting

In [79]:
df.replace(r'^\s+|\s+$', '', regex=True, inplace=True)

,name,business_type,state_registered,street_registered,city_registered,zip5_registered,state_physical,street_physical,city_physical,zip5_physical,filing_number,public
0,! ! ! APPLE IPAD & ANDROID TABLET TUTORING ! ! !,CORPORATION,CA,"10100 SANTA MONICA BLVD, # 300",LOS ANGELES,90067.0,CA,"10100 SANTA MONICA BLVD, # 300",LOS ANGELES,90067.0,C3110952,N
1,! ! 0 1 AGLC DUI AND DEFENSIVE DRIVING LLC,LLC,GA,"3235 SATELLITE BLVD., BLD. 400 STE.300",DULUTH,30032.0,GA,"3235 SATELLITE BLVD., BLD. 400",DULUTH,30096.0,15094100,N
2,! ! 0 1 ATLANTA AREA DUI AND DRIVING SCHOOL LLC,LLC,GA,4493 ASHEILA COURT,LILBURN,30047.0,GA,"3845 NORTH DRUID HILLS RD, SUITE 106",DECATUR,30033.0,10078480,N
3,! 0 $294 1 DAY MT LLC,LLC,MT,NaN,NaN,59901.0,NaN,NaN,NaN,NaN,C267413,N
4,"! ACHIEVE SUCCESS, LLC",LLC,FL,100 NE 95 ST,MIAMI SHORES,33138.0,FL,100 NE 95 ST,MIAMI SHORES,33138.0,L09000025190,N
...,...,...,...,...,...,...,...,...,...,...,...,...
995,"""FRENCH CAFE"" - OUTDOOR WEDDINGS, LIMITED LIAB...",LLC,FL,1181 SHERBROOK DR,DELTONA LAKE,32725.0,FL,1181 SHERBROOK DR,DELTONA LAKES,32725.0,L20000114507,N
996,"""FRESH COAT"" PAINTING AND HOME DECOR LLC",LLC,FL,"1942 HARDEE, STREET",JACKSONVILLE,32209.0,FL,"1942 HARDEE, STREET N/A",JACKSONVILLE,32209.0,L18000009778,N
997,"""FRESH START"" CGB (CAN'T GO BACK) INC.",NONPROFIT,FL,11885 BLUE STAR HWY,MT PLEASANT,32352.0,FL,11885 BLUE STAR HWY,MT PLEASANT,32352.0,N11000006767,N
998,"""FRIDA FOR PETS LLC""",LLC,FL,7025 NW 104 CT,MEDLEY,33178.0,FL,7025 NW 104 CT,MEDLEY,33178.0,L20000065516,N


### 5.2 Business Names

Business names contained inconsistent capitalization, whitespace,
and different representations of legal entity types such as LLC and INC.

Entity suffixes were normalized while ambiguous punctuation was intentionally preserved.

In [80]:
name_before = df["name"].copy()

In [81]:
df["name"] = df["name"].apply(clean_business_name)

In [82]:
name_changes = pd.DataFrame({
    "Before": name_before,
    "After": df["name"]
})

In [83]:
name_changes = name_changes[
    name_changes["Before"].notna()
    & name_changes["After"].notna()
    & (name_changes["Before"] != name_changes["After"])
]

In [84]:
name_changes.head(10)

,Before,After
41,!MPACT CITY INC.,!MPACT CITY INC.
52,!UNA MAS! INC,!UNA MAS! INC
74,""" BABE RUTH"" HOME RUN CONFECTION CORPORATION",""" BABE RUTH"" HOME RUN CONFECTION CORP"
94,""" EVVY"" CULTURAL INTERCHANGE, INCORPORATED",""" EVVY"" CULTURAL INTERCHANGE, INC"
118,""" MANDY THE BEECHARMER "" L.L.C.",""" MANDY THE BEECHARMER "" LLC."
120,""" MBCS COMPANY, L.L.C """,""" MBCS COMPANY, LLC """
131,""" PIRATES TRUCKING LIMITED LIABILITY COMPANY """,""" PIRATES TRUCKING LLC """
147,""" VITEC CONSULTING GROUP, INCORPORATED.""",""" VITEC CONSULTING GROUP, INC."""
149,""" WELDONE"" FAMILY LAUNDRY, INCORPORATED",""" WELDONE"" FAMILY LAUNDRY, INC"
150,""" WITNESS IN WRITING "" MINISTRY CORPORATION",""" WITNESS IN WRITING "" MINISTRY CORP"


### 5.3 Business Types

Business type values were normalized to a predefined set of valid categories. Values outside the expected categories were treated as missing.

In [85]:
df["business_type"].value_counts(dropna=False)

business_type
CORPORATION            443
LLC                    370
NONPROFIT              135
DBA                     38
PARTNERSHIP              7
SOLE PROPRIETORSHIP      7
Name: count, dtype: int64

In [86]:
df["business_type"] = df["business_type"].apply(clean_business_type)

In [87]:
df["business_type"].value_counts(dropna=False)

business_type
CORPORATION            443
LLC                    370
NONPROFIT              135
DBA                     38
PARTNERSHIP              7
SOLE PROPRIETORSHIP      7
Name: count, dtype: int64

### 5.4 State Abbreviations

State fields were expected to contain valid two-letter U.S. postal abbreviations.

Invalid values (such as `DA`) were converted to missing values when they could not be reliably corrected.

In [88]:
df["state_registered"].value_counts(dropna=False).sort_values()

state_registered
AR      1
ID      1
DA      2
NC      3
DE      3
MS      3
AK      4
TX      4
AL      4
ND      5
CA      6
VA      8
WV     11
MT     13
OR     21
CT     28
WA     30
CO     35
MO     43
GA     49
IN     56
PA    133
OH    171
NY    172
FL    194
Name: count, dtype: int64

In [89]:
df["state_physical"].value_counts(dropna=False).sort_values()

state_physical
AR       1
WY       1
TN       1
AL       1
MA       2
NC       2
TX       4
WV       4
AK       4
CA       5
MT       6
VA       8
OR      16
CT      23
PA      24
WA      30
CO      33
NY      36
OH      43
GA      50
IN      55
FL     197
NaN    454
Name: count, dtype: int64

In [90]:
df["state_registered"] = df["state_registered"].apply(clean_state_postal_abbr)

df["state_physical"] = df["state_physical"].apply(clean_state_postal_abbr)

In [91]:
df["state_registered"].value_counts(dropna=False).sort_values()

state_registered
AR       1
ID       1
NaN      2
NC       3
DE       3
MS       3
AK       4
TX       4
AL       4
ND       5
CA       6
VA       8
WV      11
MT      13
OR      21
CT      28
WA      30
CO      35
MO      43
GA      49
IN      56
PA     133
OH     171
NY     172
FL     194
Name: count, dtype: int64

In [92]:
df["state_physical"].value_counts(dropna=False).sort_values()

state_physical
AR       1
WY       1
TN       1
AL       1
MA       2
NC       2
TX       4
WV       4
AK       4
CA       5
MT       6
VA       8
OR      16
CT      23
PA      24
WA      30
CO      33
NY      36
OH      43
GA      50
IN      55
FL     197
NaN    454
Name: count, dtype: int64

### 5.5 Cities

City values contained inconsistent capitalization, whitespace, abbreviations, and a small number of identifiable spelling errors.

A conservative normalization strategy was applied. Ambiguous geographic values were not forcefully remapped.

In [93]:
city_before = df["city_registered"].copy()

In [94]:
df["city_registered"] = df["city_registered"].apply(clean_us_city)

df["city_physical"] = df["city_physical"].apply(clean_us_city)

In [95]:
city_changes = pd.DataFrame({
    "Before": city_before,
    "After": df["city_registered"]
})

In [96]:
city_changes = city_changes[
    city_changes["Before"].notna()
    & city_changes["After"].notna()
    & (city_changes["Before"] != city_changes["After"])
]

In [97]:
city_changes.head(10)

,Before,After
47,ST. LOUIS,SAINT LOUIS
116,ST ALBANS,SAINT ALBANS
159,PORT ST LUCIE,PORT SAINT LUCIE
163,ST. LOUIS,SAINT LOUIS
173,UNIVERSITY PL,UNIVERSITY PLACE
336,SW RANCHES,SOUTHWEST RANCHES
417,ST ANTHONY,SAINT ANTHONY
492,ST. LOUIS,SAINT LOUIS
533,ST. LOUIS,SAINT LOUIS
542,STONEMOUNTAIN,STONE MOUNTAIN


### 5.6 Addresses

Street addresses contained inconsistent whitespace, punctuation, street suffixes, and secondary-unit designators.

The cleaning function standardizes these elements while preserving the underlying address information.

In [98]:
address_before = df["street_registered"].copy()

In [99]:
df["street_registered"] = df["street_registered"].apply(clean_address)

df["street_physical"] = df["street_physical"].apply(clean_address)

In [128]:
address_changes = pd.DataFrame({
    "Before": address_before,
    "After": df["street_registered"]
})

In [129]:
address_changes = address_changes[
    address_changes["Before"].notna()
    & address_changes["After"].notna()
    & (address_changes["Before"] != address_changes["After"])
]

In [130]:
address_changes.head(10)

,Before,After
0,"10100 SANTA MONICA BLVD, # 300","10100 SANTA MONICA BLVD, #300"
1,"3235 SATELLITE BLVD., BLD. 400 STE.300","3235 SATELLITE BLVD, BLD 400 STE300"
2,4493 ASHEILA COURT,4493 ASHEILA CT
6,823 MARY ANN DRIVE,823 MARY ANN DR
7,216 N. FAYETTE STREET,216 N FAYETTE ST
8,2129 GROVE POINT ROAD,2129 GRV PT RD
17,1609 N. NOVA RD,1609 N NOVA RD
19,60 BROAD ST SUITE 3101,60 BROAD ST STE 3101
21,4607 MEADOW AVE,4607 MDW AVE
29,4549 REVENUE TRAIL,4549 REVENUE TRL


### 5.7 ZIP Codes

ZIP codes were imported as numeric values, which caused leading zeros to be lost during type inference. For example, `01581` was represented as `1581.0`.

Because ZIP codes are identifiers rather than quantities, they should be stored as strings. Four-digit values were therefore zero-padded to five digits, while invalid values were treated as missing.

In [103]:
mask = (
    ~df["zip5_physical"].between(10000, 99999)
    & df["zip5_physical"].notna()
)

In [104]:
zip_before = df.loc[mask, "zip5_physical"].copy()

In [105]:
df["zip5_registered"] = df["zip5_registered"].apply(clean_zip_code)

df["zip5_physical"] = df["zip5_physical"].apply(clean_zip_code)

In [106]:
zip_after = df.loc[mask, "zip5_physical"]

In [131]:
pd.DataFrame({
    "Before": zip_before,
    "After": zip_after
}).head(10)

,Before,After
75,6770.0,06770
165,6360.0,06360
174,6437.0,06437
205,6776.0,06776
227,6489.0,06489
269,1581.0,01581
318,6611.0,06611
363,6371.0,06371
434,6053.0,06053
437,6320.0,06320


## 6. Validation

### 6.1 ZIP/State Consistency

ZIP codes and state abbreviations were checked for consistency using a reference ZIP-code dataset.

Because the reference dataset may not contain every valid U.S. ZIP code, the validation was performed based on ZIP code prefixes rather than exact ZIP code matches. The prefix was used to determine whether a ZIP code is geographically consistent with its corresponding state.

The validation results were stored in two additional columns:

- `registered_zip_state_validation`

- `physical_zip_state_validation`

Records were classified as valid, invalid, or missing based on the available ZIP and state information.

#### 6.1.1 Loading the Validation Dataset

In [108]:
validation_df = pd.read_csv(
    EXTERNAL_DATA_DIR / "zip_codes_database.csv",
    dtype={"ZipCode": str},
    usecols=["ZipCode", "State"]
)

In [109]:
validation_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 41684 entries, 0 to 41683
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype
---  ------   --------------  -----
 0   ZipCode  41684 non-null  str  
 1   State    41684 non-null  str  
dtypes: str(2)
memory usage: 651.4 KB


In [110]:
validation_df.head()

,ZipCode,State
0,00501,NY
1,00544,NY
2,00601,PR
3,00602,PR
4,00603,PR


In [111]:
validation_df.rename(
    columns={
        "ZipCode": "zip_code",
        "State": "state"
    },
    inplace=True
)

#### 6.1.2 Validation

In [112]:
df["registered_zip_state_validation"] = validate_zip_state(
    df=df,
    validation_df=validation_df,
    zip_col="zip5_registered",
    state_col="state_registered",
    ref_zip_col="zip_code",
    ref_state_col="state"
)

In [113]:
df["physical_zip_state_validation"] = validate_zip_state(
    df=df,
    validation_df=validation_df,
    zip_col="zip5_physical",
    state_col="state_physical",
    ref_zip_col="zip_code",
    ref_state_col="state"
)

In [114]:
df["registered_zip_state_validation"].value_counts(dropna=False)

registered_zip_state_validation
valid      606
missing    388
invalid      6
Name: count, dtype: int64

In [115]:
df["physical_zip_state_validation"].value_counts(dropna=False)

physical_zip_state_validation
missing    520
valid      478
invalid      2
Name: count, dtype: int64

In [116]:
df.loc[
    df["registered_zip_state_validation"] == "invalid",
    ["zip5_registered", "state_registered"]
].head(10)

,zip5_registered,state_registered
5,99201,FL
269,98501,ID
440,30075,FL
444,98501,DE
522,37217,FL
574,98501,DE


In [117]:
df.loc[
    df["physical_zip_state_validation"] == "invalid",
    ["zip5_physical", "state_physical"]
].head(10)

,zip5_physical,state_physical
283,34104,AL
343,20904,IN


### 6.2 Final Data Quality Checks

In [118]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype
---  ------                           --------------  -----
 0   name                             1000 non-null   str  
 1   business_type                    1000 non-null   str  
 2   state_registered                 998 non-null    str  
 3   street_registered                630 non-null    str  
 4   city_registered                  632 non-null    str  
 5   zip5_registered                  612 non-null    str  
 6   state_physical                   546 non-null    str  
 7   street_physical                  598 non-null    str  
 8   city_physical                    770 non-null    str  
 9   zip5_physical                    485 non-null    str  
 10  filing_number                    989 non-null    str  
 11  public                           811 non-null    str  
 12  registered_zip_state_validation  1000 non-null   str  
 13  

In [119]:
df.isna().sum().sort_values(ascending=False)

zip5_physical                      515
state_physical                     454
street_physical                    402
zip5_registered                    388
street_registered                  370
city_registered                    368
city_physical                      230
public                             189
filing_number                       11
state_registered                     2
name                                 0
business_type                        0
registered_zip_state_validation      0
physical_zip_state_validation        0
dtype: int64

In [120]:
df.duplicated().sum()

np.int64(0)

In [121]:
df["business_type"].value_counts(dropna=False)

business_type
CORPORATION            443
LLC                    370
NONPROFIT              135
DBA                     38
PARTNERSHIP              7
SOLE PROPRIETORSHIP      7
Name: count, dtype: int64

In [122]:
df["state_registered"].value_counts(dropna=False).sort_index()

state_registered
AK       4
AL       4
AR       1
CA       6
CO      35
CT      28
DE       3
FL     194
GA      49
ID       1
IN      56
MO      43
MS       3
MT      13
NC       3
ND       5
NY     172
OH     171
OR      21
PA     133
TX       4
VA       8
WA      30
WV      11
NaN      2
Name: count, dtype: int64

## 7. Exporting the Cleaned Dataset

After completing the cleaning and validation steps, the final DataFrame was exported as a CSV file for further analysis and reuse.

In [123]:
df.to_csv(PROCESSED_DATA_DIR / "cleaned_us_businesses.csv", index=False)

The exported file was reloaded to verify that it was written successfully and preserved the expected dataset dimensions.

In [124]:
cleaned_df = pd.read_csv(
    PROCESSED_DATA_DIR / "cleaned_us_businesses.csv",
    dtype={
        "zip5_registered": "string",
        "zip5_physical": "string"
    }
)

In [125]:
df.shape == cleaned_df.shape

True

In [126]:
cleaned_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 14 columns):
 #   Column                           Non-Null Count  Dtype 
---  ------                           --------------  ----- 
 0   name                             1000 non-null   str   
 1   business_type                    1000 non-null   str   
 2   state_registered                 998 non-null    str   
 3   street_registered                630 non-null    str   
 4   city_registered                  632 non-null    str   
 5   zip5_registered                  612 non-null    string
 6   state_physical                   546 non-null    str   
 7   street_physical                  598 non-null    str   
 8   city_physical                    770 non-null    str   
 9   zip5_physical                    485 non-null    string
 10  filing_number                    989 non-null    str   
 11  public                           811 non-null    str   
 12  registered_zip_state_validation  1000 non-null

## 8. Conclusion

The dataset was cleaned through a structured and conservative data-cleaning workflow.

The process addressed missing and uninformative columns, inconsistent state and business-type values, city-name formatting, ZIP-code representation, street-address formatting, and business-name normalization.

ZIP and state combinations were additionally checked against an external reference dataset.

Ambiguous values that could not be reliably corrected were preserved or converted to missing values rather than being modified based on unsupported assumptions.

The final dataset is therefore more consistent and suitable for downstream analysis while retaining potentially meaningful information from the original records.